SMOTE is being applied to the normalized data
ie. base -> normalized -> SMOTE

In [4]:
import numpy as np
import joblib
import os
from sklearn.metrics import classification_report, accuracy_score

# File paths
SAVE_PATH = "../processed_data/SDHAR/"
LSTM_DATA_FILE = os.path.join(SAVE_PATH, "lstm_smote_data.npz")
TREE_DATA_FILE = os.path.join(SAVE_PATH, "tree_smote_data.npz")

# Save paths
MODEL_SAVE_PATH = "../models/SDHAR/"
if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)

activity_names = ["BATHROOM ACTIVITY", "CHORES", "COOK", "DISHWASHING", "DRESS", "EAT", "LAUNDRY",
                  "MAKE SIMPLE FOOD", "OUT HOME", "PET", "READ", "RELAX", "SHOWER", "SLEEP",
                  "TAKE MEDS", "WATCH TV", "WORK", "OTHER"]

# LSTM

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

print(f"Loading data from {LSTM_DATA_FILE}...")
# 1. Load the preprocessed LSTM data
data = np.load(LSTM_DATA_FILE)
X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']
y_test_1d_labels = data['y_test_1d_labels'] # For the final report

# Get shape info from the loaded data
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]
n_classes = y_train.shape[1]

print(f"  - Training X shape: {X_train.shape}")
print(f"  - Training y shape: {y_train.shape}")
print(f"  - Test X shape: {X_test.shape}")
print(f"  - Test y shape: {y_test.shape}")

print("\nBuilding the LSTM model...")
model_lstm = Sequential([
    LSTM(64, input_shape=(n_timesteps, n_features), return_sequences=True),
    Dropout(0.5),
    LSTM(64),
    Dropout(0.5),
    Dense(n_classes, activation='softmax')
])

model_lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_lstm.summary()

print("\nTraining the LSTM model on SMOTE data...")
history = model_lstm.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)
model_lstm.save(os.path.join(MODEL_SAVE_PATH, 'LSTM_SMOTE_final.keras'))

print("\nEvaluating the LSTM model on the *original* (unbalanced) test set...")
loss, accuracy = model_lstm.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

y_pred_probs = model_lstm.predict(X_test)
y_pred_labels = np.argmax(y_pred_probs, axis=1) # Convert one-hot back to 1D

print("\nClassification Report (LSTM with SMOTE):")
# Compare the 1D predicted labels with the 1D true labels
print(classification_report(y_test_1d_labels, y_pred_labels, target_names=activity_names))

Loading data from ../processed_data/SDHAR/lstm_smote_data.npz...
  - Training X shape: (45814, 60, 41)
  - Training y shape: (45814, 18)
  - Test X shape: (13494, 60, 41)
  - Test y shape: (13494, 18)

Building the LSTM model...


C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 64)         │        27,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 18)             │         1,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,330 (239.57 KB)

 Trainable params: 61,330 (239.57 KB)

 Non-trainable params: 0 (0.00 B)


Training the LSTM model on SMOTE data...
Epoch 1/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 61s 145ms/step - accuracy: 0.6388 - loss: 1.2076 - val_accuracy: 0.0000e+00 - val_loss: 5.4184
Epoch 2/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 35s 107ms/step - accuracy: 0.8082 - loss: 0.6329 - val_accuracy: 0.0279 - val_loss: 5.4586
Epoch 3/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 34s 105ms/step - accuracy: 0.8470 - loss: 0.4985 - val_accuracy: 0.1395 - val_loss: 5.6588
Epoch 4/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 36s 113ms/step - accuracy: 0.8705 - loss: 0.4198 - val_accuracy: 0.1912 - val_loss: 5.8726
Epoch 5/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 32s 100ms/step - accuracy: 0.8837 - loss: 0.3814 - val_accuracy: 0.2139 - val_loss: 5.8773
Epoch 6/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 30s 92ms/step - accuracy: 0.8977 - loss: 0.3358 - val_accuracy: 0.2842 - val_loss: 5.8844
Epoch 7/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 30s 92ms/step - accuracy: 0.9067 - loss: 0.3034 - val_accuracy: 0.2254 - val_loss: 6.2276
Epoch 8/10
323/323 ━━━━━━━━━━━━━━━━━━━━

C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

# Random Forest

In [6]:
from sklearn.ensemble import RandomForestClassifier

print(f"Loading data from {TREE_DATA_FILE}...")
data = np.load(TREE_DATA_FILE)
X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']

print(f"  - Training X shape: {X_train.shape}")
print(f"  - Training y shape: {y_train.shape}")
print(f"  - Test X shape: {X_test.shape}")
print(f"  - Test y shape: {y_test.shape}")

print("\nBuilding and training the Random Forest model on SMOTE data...")
model_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

model_rf.fit(X_train, y_train)
print("  - Model training complete!")

print("\nEvaluating the RF model on the *original* (unbalanced) test set...")
y_pred = model_rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report (Random Forest with SMOTE):")
print(classification_report(y_test, y_pred, target_names=activity_names))

print("Saving the RF model...")
joblib.dump(model_rf, os.path.join(MODEL_SAVE_PATH, "RandomForest_SMOTE_final.joblib"))
print("  - Model saved successfully!")

Loading data from ../processed_data/SDHAR/tree_smote_data.npz...
  - Training X shape: (45814, 2460)
  - Training y shape: (45814,)
  - Test X shape: (13494, 2460)
  - Test y shape: (13494,)

Building and training the Random Forest model on SMOTE data...
  - Model training complete!

Evaluating the RF model on the *original* (unbalanced) test set...

Test Accuracy: 97.88%

Classification Report (Random Forest with SMOTE):
                   precision    recall  f1-score   support

BATHROOM ACTIVITY       0.94      0.94      0.94       400
           CHORES       0.94      0.85      0.89        74
             COOK       0.89      0.95      0.92       134
      DISHWASHING       0.85      0.94      0.89        18
            DRESS       0.65      0.62      0.63        32
              EAT       0.94      0.96      0.95       641
          LAUNDRY       1.00      1.00      1.00         2
 MAKE SIMPLE FOOD       0.84      0.85      0.85        89
         OUT HOME       1.00      0.99    

# Decision Tree

In [7]:
from sklearn.tree import DecisionTreeClassifier

print(f"Loading data from {TREE_DATA_FILE}...")
data = np.load(TREE_DATA_FILE)
X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']

print(f"  - Training X shape: {X_train.shape}")
print(f"  - Training y shape: {y_train.shape}")
print(f"  - Test X shape: {X_test.shape}")
print(f"  - Test y shape: {y_test.shape}")

print("\nBuilding and training the Decision Tree model on SMOTE data...")
model_dt = DecisionTreeClassifier(random_state=42)

model_dt.fit(X_train, y_train)
print("  - Model training complete!")

print("\nEvaluating the DT model on the *original* (unbalanced) test set...")
y_pred = model_dt.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report (Decision Tree with SMOTE):")
print(classification_report(y_test, y_pred, target_names=activity_names))

print("Saving the DT model...")
joblib.dump(model_dt, os.path.join(MODEL_SAVE_PATH, "DecisionTree_SMOTE_final.joblib"))
print("  - Model saved successfully!")

Loading data from ../processed_data/SDHAR/tree_smote_data.npz...
  - Training X shape: (45814, 2460)
  - Training y shape: (45814,)
  - Test X shape: (13494, 2460)
  - Test y shape: (13494,)

Building and training the Decision Tree model on SMOTE data...
  - Model training complete!

Evaluating the DT model on the *original* (unbalanced) test set...

Test Accuracy: 94.66%

Classification Report (Decision Tree with SMOTE):
                   precision    recall  f1-score   support

BATHROOM ACTIVITY       0.87      0.87      0.87       400
           CHORES       0.73      0.61      0.66        74
             COOK       0.71      0.82      0.76       134
      DISHWASHING       0.68      0.83      0.75        18
            DRESS       0.30      0.34      0.32        32
              EAT       0.88      0.87      0.87       641
          LAUNDRY       0.50      1.00      0.67         2
 MAKE SIMPLE FOOD       0.50      0.58      0.54        89
         OUT HOME       1.00      0.98    

# Sampled-Adjusted Models Analysis

Both the DT and RF models decievingly perform worse when over/undersampling minority classes than without. Although their overall accuracies are lower, the key is in the macro avgs. Specifically for the Random Forest model, we saw significant jumps (ie 77%->91%) in each category. The RF model also currently stands out as the leading model.

Unfortunately, the LSTM model improved in areas like recal weighted accuracy, but completely dropped the ball on the "Other" category as well as some other accuracies. If we can find a fix to this, perhaps the model can be improved.

Answer: Why SMOTE did not work -- we need to use another approach
